# Stage 2 Notebook 31 - Exp2Z Mask aux + Kendall uncertainty weighting

**Why this exists.** Companion to Exp2Y (NB30 fixed-lambda fix). Same diagnosis: grad_norm_calibration's per-epoch recomputation creates 4x lambda swings that destabilize joint training.

Exp2Y's fix is heuristic: lock lambda at 1.0. Exp2Z's fix is theoretically grounded -- enable Kendall et al. 2018 uncertainty-based multi-task weighting. The model **learns** log-variance parameters `sigma_lane`, `sigma_det` and weighs the joint loss as:

```
L_total = (1 / (2 * sigma_lane^2)) * L_lane + log(sigma_lane)
        + (1 / (2 * sigma_det^2))  * L_det  + log(sigma_det)
```

The model itself decides how much weight to give each task. The `log(sigma)` term prevents trivial solution of sigma -> infinity. Monotonic by construction; oscillation is structurally impossible.

Implementation: the codebase already has `UncertaintyMultiTaskLoss` in `stage2/fusion/losses.py`. We just enable it via `loss.use_uncertainty: true`.

Single-knob change vs Exp2W: `use_uncertainty: false -> true`. Plus the same warmup/duration tweaks as Exp2Y for consistency. Independent of Exp2Y -- run either or both.

Reference: Kendall, Gal, Cipolla 2018 'Multi-Task Learning Using Uncertainty to Weigh Losses for Scene Geometry and Semantics' (CVPR).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 15-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp26_rmt_gca_mask_uncertainty_weighting_joint_smoke.log
OK exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.0210 det_loss=3.5114 grad_cos=-0.0614 lambda_lane=0.2049
  gate_stats={'gate/det_mean': 0.4987221956253052, 'gate/lane_mean': 0.5010656118392944, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short15'
    EPOCHS = 15
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar --epochs 15 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yam

0

## What to watch in Exp2Z training

Pass criteria at epoch 15:
- **`mtl/log_var_0` and `mtl/log_var_1`** logged in metrics JSON, evolving smoothly. These are the learned `log(sigma)` values; track to confirm uncertainty weighting is active.
- **`val/lane/decoded_f1 >= 0.07`**: stable monotonic training should beat Exp2W's 0.043.
- **`val/matched_line_iou >= 0.20`**.
- **`train_total` decreases monotonically** -- no sawtooth from lambda swings.

Comparison with Exp2Y (fixed lambda):
- If both work similarly: lambda stability was the issue, either fix is fine.
- If Exp2Z significantly beats Exp2Y: learned task balancing matters more than just stable weights.
- If Exp2Z fails but Exp2Y works: uncertainty weighting has its own pathology in this regime; revert to fixed.